# Chat with Your PDFs -- runnable notebook

This notebook is a Colab/Kaggle/Binder-friendly version of the **Chat with Your PDFs** project from the
[Python & Data Analysis course](https://github.com/abderrahim-lectures/python-data-analysis-course). It mirrors the
real, runnable example at [`examples/chat-with-pdfs/`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/chat-with-pdfs):
generate (or upload) a few sample PDFs, chunk them page by page, embed the chunks locally, then retrieve and generate
a page-cited answer with a free-tier LLM.

Notebook sessions are ephemeral, so this uses `getpass()` to enter your API key each run instead of a `.env` file.

In [ ]:
!pip install pypdf sentence-transformers numpy openai reportlab -q

## Generate sample PDFs

The real example ships with three ready-made sample PDFs. Since this notebook may be running on a fresh Colab/Kaggle
VM with no repo checked out, this cell regenerates the same three PDFs locally with `reportlab`. If you'd rather use
your own PDFs, skip this cell and upload files into a `pdfs/` folder instead (Colab: the file browser on the left;
Kaggle: Add Data).

In [ ]:
from pathlib import Path
from reportlab.lib.pagesizes import LETTER
from reportlab.lib.units import inch
from reportlab.pdfgen import canvas

PDFS_DIR = Path("pdfs")
PDFS_DIR.mkdir(exist_ok=True)


def wrap_text(text, width=90):
    words, lines, current = text.split(), [], ""
    for word in words:
        candidate = f"{current} {word}".strip()
        if len(candidate) > width:
            lines.append(current)
            current = word
        else:
            current = candidate
    if current:
        lines.append(current)
    return lines


def write_pdf(path, title, pages):
    c = canvas.Canvas(str(path), pagesize=LETTER)
    width, height = LETTER
    for page_lines in pages:
        c.setFont("Helvetica-Bold", 16)
        c.drawString(1 * inch, height - 1 * inch, title)
        c.setFont("Helvetica", 11)
        y = height - 1.5 * inch
        for line in page_lines:
            for wrapped in wrap_text(line):
                c.drawString(1 * inch, y, wrapped)
                y -= 0.25 * inch
            y -= 0.15 * inch
        c.showPage()
    c.save()


write_pdf(PDFS_DIR / "employee-handbook.pdf", "Northwind Traders -- Employee Handbook", [
    ["Section 1: Time Off",
     "Full-time employees accrue 18 days of paid time off per year, credited at the start of each quarter. "
     "Unused days roll over up to a maximum of 10 days into the following year.",
     "Requests for time off must be submitted at least 5 business days in advance through the HR portal, "
     "except in the case of documented medical emergencies."],
    ["Section 2: Remote Work",
     "Employees may work remotely up to 3 days per week with manager approval. Fully remote arrangements "
     "require a written agreement renewed annually with HR."],
    ["Section 3: Expense Reimbursement",
     "Business expenses under $75 can be self-approved and submitted through the expense portal within 30 "
     "days of purchase. Expenses above $75 require prior manager approval."],
])

write_pdf(PDFS_DIR / "product-warranty.pdf", "Aurora Blender 3000 -- Warranty Guide", [
    ["Coverage Period",
     "The Aurora Blender 3000 is covered by a 2-year limited warranty from the date of original purchase, "
     "covering defects in materials and workmanship under normal household use."],
    ["What Is Not Covered",
     "The warranty does not cover damage from misuse, unauthorized repairs, commercial use, or normal wear "
     "of the blade assembly, which is considered a consumable part."],
    ["How to File a Claim",
     "To file a warranty claim, register your product and contact support with your order number and a "
     "description of the defect. Approved claims are resolved within 10 business days."],
])

write_pdf(PDFS_DIR / "city-permit-guide.pdf", "Riverside City -- Home Renovation Permit Guide", [
    ["When You Need a Permit",
     "A building permit is required for any structural changes, electrical rewiring, new plumbing lines, or "
     "additions over 120 square feet. Cosmetic work such as painting or flooring does not require a permit."],
    ["Application Process",
     "Permit applications are submitted online and typically reviewed within 15 business days. A licensed "
     "contractor's information is required for electrical and plumbing permits."],
    ["Fees and Inspections",
     "Permit fees start at $85 for projects under $5,000. At least one inspection is required before a "
     "permit is closed out."],
])

print(f"Wrote {len(list(PDFS_DIR.glob(chr(42) + \'.pdf\')))} PDFs to {PDFS_DIR}/")

## Load and chunk the PDFs, keeping page numbers

Each chunk keeps its source filename and 1-indexed page number attached -- this is what makes citations possible
later.

In [ ]:
from pypdf import PdfReader

TARGET_CHUNK_SIZE = 500


def split_into_paragraphs(text):
    paragraphs = [p.strip() for p in text.split("\n\n")]
    paragraphs = [p for p in paragraphs if p]
    if len(paragraphs) <= 1:
        paragraphs = [p.strip() for p in text.split("\n")]
        paragraphs = [p for p in paragraphs if p]
    return paragraphs


def merge_short_paragraphs(paragraphs, target_size):
    chunks, current = [], ""
    for paragraph in paragraphs:
        if current and len(current) + len(paragraph) > target_size:
            chunks.append(current)
            current = paragraph
        else:
            current = f"{current}\n\n{paragraph}" if current else paragraph
    if current:
        chunks.append(current)
    return chunks


def load_chunks():
    chunks = []
    for path in sorted(PDFS_DIR.glob("*.pdf")):
        reader = PdfReader(str(path))
        for page_index, page in enumerate(reader.pages):
            text = page.extract_text() or ""
            for chunk_text in merge_short_paragraphs(split_into_paragraphs(text), TARGET_CHUNK_SIZE):
                chunks.append({"text": chunk_text, "source": path.name, "page": page_index + 1})
    return chunks


chunks = load_chunks()
print(f"Loaded {len(chunks)} chunks")
for c in chunks[:3]:
    print(f"  [{c[\'source\']} p{c[\'page\']}] {c[\'text\'][:80].replace(chr(10), \' \')}...")

## Embed the chunks locally

`all-MiniLM-L6-v2` runs entirely on CPU, needs no API key, and costs nothing -- same model used in the local example.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)
texts = [c["text"] for c in chunks]
embeddings = model.encode(texts, normalize_embeddings=True)
print(f"Embedded {embeddings.shape[0]} chunks into {embeddings.shape[1]}-dim vectors")

## Retrieve relevant chunks for a question

Cosine similarity via a single dot product, since every vector was normalized to length 1 above.

In [ ]:
def retrieve(question, top_k=4):
    question_vector = model.encode([question], normalize_embeddings=True)[0]
    similarities = embeddings @ question_vector
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [{**chunks[i], "score": float(similarities[i])} for i in top_indices]


for r in retrieve("How many days of paid time off do employees get?"):
    print(f"{r['score']:.3f}  [{r['source']} p{r['page']}]  {r['text'][:80]}...")

## Pick a provider and enter your API key

**You're free to use whichever provider you like.** GitHub Models is the suggested default below since it needs no
separate signup (you already have a GitHub account), but Gemini, Groq, Mistral, Cerebras, and OpenRouter all have
workable free tiers too. `getpass()` keeps the key out of the notebook's saved output.

In [ ]:
import os
from getpass import getpass

# One of: "github" (default), "gemini", "groq", "mistral", "cerebras", "openrouter"
PROVIDER = "github"

ENV_VAR_BY_PROVIDER = {
    "github": "GITHUB_TOKEN",
    "gemini": "GOOGLE_API_KEY",
    "groq": "GROQ_API_KEY",
    "mistral": "MISTRAL_API_KEY",
    "cerebras": "CEREBRAS_API_KEY",
    "openrouter": "OPENROUTER_API_KEY",
}

env_var = ENV_VAR_BY_PROVIDER[PROVIDER]
os.environ[env_var] = getpass(f"Enter your {PROVIDER} API key ({env_var}): ")

## Retrieve + generate a cited answer

GitHub Models exposes an OpenAI-compatible API, so the plain `openai` client library works for it without any extra
package. If you picked a different provider above, swap in that provider's own client here -- see the
[lesson's tip](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/docs/projects/chat-with-pdfs/index.md)
for the exact substitution, same pattern as the AI Agent project.

In [ ]:
from openai import OpenAI

PROMPT_TEMPLATE = """Answer the question using ONLY the context below. If the
context doesn\'t contain the answer, say so -- do not make something up.

Every fact you use MUST be followed by a citation in the form
(source, page N), taken from the [source, page N] tag on the context chunk
it came from. If your answer draws on more than one chunk, cite each one.

Context:
{context}

Question: {question}

Answer:"""


def build_prompt(question, retrieved_chunks):
    context = "\n\n".join(f"[{c['source']}, page {c['page']}] {c['text']}" for c in retrieved_chunks)
    return PROMPT_TEMPLATE.format(context=context, question=question)


def ask(question, top_k=4):
    retrieved_chunks = retrieve(question, top_k=top_k)
    prompt = build_prompt(question, retrieved_chunks)

    client = OpenAI(
        api_key=os.environ["GITHUB_TOKEN"],
        base_url="https://models.github.ai/inference",
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",  # confirm this still has a free tier before relying on it
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content


print(ask("How many days of paid time off do employees get?"))

## Try your own questions

Ask about anything covered in the three sample PDFs -- time off, remote work, the blender warranty, or city permits
-- and check that the citations actually point at the right document and page.

In [ ]:
print(ask("What voids the blender warranty?"))